## Identifying Hard Optimization Problems in Refractory Alloy Dataset

The purpose of this notebook is to analyze a dataset to identify difficult optimization problems. This is done in 4 steps:
1. Correlation analysis (objective conflict)
2. Exploratory Landscape Analysis (ELA)
3. Pareto front characteristics
4. Combined difficulty scoring

In [ ]:
"""
Step 1: Load Data and Setup
Loads the refractory alloy dataset and identifies input/output columns
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def load_and_prepare_data(filepath='./data/1000_refractory_alloy_calphad_modeled_data.xlsx'):
    """Load and prepare the dataset"""
    
    print("=" * 80)
    print("LOADING AND PREPARING DATA")
    print("=" * 80)
    
    # Load data
    df = pd.read_excel(filepath)
    
    # Define input and output columns
    input_cols = ["Ti", "V", "Nb", "Mo", "Hf", "Ta", "W"]
    columns_to_remove = ["System Type", "Unnamed: 0.2", "System Size", 
                         "EQ 700C MAX BCC", "EQ 800C MAX BCC", "EQ 1000C MAX BCC", 
                         "EQ 25C MAX NON-BCC"]
    
    
    df = df.iloc[:, 1:]
    
    # Get all property columns (everything except inputs and columns to remove)
    all_cols = set(df.columns)
    property_cols = list(all_cols - set(input_cols) - set(columns_to_remove))
    property_cols = [col for col in property_cols if col in df.columns]  # Ensure they exist
    
    print(f"\nDataset shape: {df.shape}")
    print(f"Input columns ({len(input_cols)}): {input_cols}")
    print(f"\nProperty columns ({len(property_cols)}):")
    for i, col in enumerate(property_cols[:15], 1):
        print(f"  {i}. {col}")
    if len(property_cols) > 15:
        print(f"  ... and {len(property_cols) - 15} more")
    print(f"\nTotal property columns: {len(property_cols)}")
    
    return df, input_cols, property_cols

df, input_cols, property_cols = load_and_prepare_data()

# Save for next steps
np.savez('data_config.npz', 
            input_cols=input_cols, 
            property_cols=property_cols,
            allow_pickle=True)
df.to_pickle('df_prepared.pkl')

print("\n✓ Data prepared and saved to 'df_prepared.pkl' and 'data_config.npz'")

In [ ]:
"""
Step 2: Correlation Analysis
Computes Spearman correlations between all property pairs to identify conflicting objectives
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from itertools import combinations

def perform_correlation_analysis(df, property_cols):
    """Analyze correlations between all property pairs"""
    
    print("=" * 80)
    print("STEP 2: CORRELATION ANALYSIS")
    print("=" * 80)
    
    # Compute Spearman correlation matrix for all properties
    property_data = df[property_cols].dropna(axis=1, how='all')  # Remove columns with all NaN
    corr_matrix = property_data.corr(method='spearman')
    
    # Find all pairs and their correlations
    pairs_correlations = []
    for col1, col2 in combinations(property_data.columns, 2):
        corr_val = corr_matrix.loc[col1, col2]
        if not np.isnan(corr_val):
            pairs_correlations.append({
                'Property_1': col1,
                'Property_2': col2,
                'Correlation': corr_val,
                'Abs_Correlation': abs(corr_val),
                'Conflict_Score': -corr_val if corr_val < 0 else 0  # Higher = more conflicting
            })
    
    pairs_df = pd.DataFrame(pairs_correlations)
    
    # Sort by most conflicting (most negative correlation)
    pairs_df_sorted = pairs_df.sort_values('Correlation', ascending=True)
    
    print("\nTop 20 Most Conflicting Objective Pairs (Most Negative Correlation):")
    print("=" * 80)
    print(pairs_df_sorted.head(20).to_string(index=False))
    
    # Visualize top conflicts
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram of correlations
    axes[0].hist(pairs_df['Correlation'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0, color='red', linestyle='--', label='No correlation')
    axes[0].axvline(x=-0.7, color='orange', linestyle='--', label='High conflict threshold')
    axes[0].set_xlabel('Spearman Correlation')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Objective Correlations')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Top 15 most conflicting pairs
    top_conflicts = pairs_df_sorted.head(15)
    y_pos = np.arange(len(top_conflicts))
    labels = [f"{row['Property_1'][:20]}\nvs\n{row['Property_2'][:20]}" 
              for _, row in top_conflicts.iterrows()]
    axes[1].barh(y_pos, top_conflicts['Correlation'], color='crimson', alpha=0.7)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(labels, fontsize=7)
    axes[1].set_xlabel('Correlation')
    axes[1].set_title('Top 15 Most Conflicting Pairs')
    axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('correlation_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved correlation analysis plot to 'correlation_analysis.png'")
    plt.show()
    
    # Select top 10 most conflicting pairs for further analysis
    top_10_pairs = pairs_df_sorted.head(10)
    print(f"\n✓ Selected top 10 pairs for detailed analysis")
    
    return pairs_df_sorted, top_10_pairs

# Load data
df = pd.read_pickle('df_prepared.pkl')
config = np.load('data_config.npz', allow_pickle=True)
property_cols = config['property_cols'].tolist()

# Run analysis
pairs_df_sorted, top_10_pairs = perform_correlation_analysis(df, property_cols)

# Save results
pairs_df_sorted.to_csv('correlation_results.csv', index=False)
top_10_pairs.to_csv('top_10_pairs.csv', index=False)

print("\n✓ Results saved to 'correlation_results.csv' and 'top_10_pairs.csv'")

In [ ]:
# Cell 3: Step 2 - Exploratory Landscape Analysis (ELA)
print("\n" + "=" * 80)
print("STEP 2: EXPLORATORY LANDSCAPE ANALYSIS (ELA)")
print("=" * 80)

if not pflacco_available:
    print("\nSkipping ELA analysis - pflacco not installed.")
    print("Install with: pip install pflacco --break-system-packages")
    ela_results = []
else:
    ela_results = []
    X = df[input_cols].values
    
    print("\nAnalyzing landscape features for top 10 most conflicting pairs...\n")
    
    for idx, row in top_10_pairs.iterrows():
        prop1, prop2 = row['Property_1'], row['Property_2']
        
        # Get objective values
        y1 = df[prop1].values
        y2 = df[prop2].values
        
        # Remove NaN entries
        valid_idx = ~(np.isnan(y1) | np.isnan(y2))
        X_valid = X[valid_idx]
        y1_valid = y1[valid_idx]
        y2_valid = y2[valid_idx]
        
        if len(y1_valid) < 50:  # Need enough samples
            continue
        
        try:
            # Analyze landscape for each objective separately
            # Note: ELA typically works with single objectives, so we analyze each
            
            # For objective 1
            ela_meta_1 = calculate_ela_meta(X_valid, y1_valid)
            ela_dist_1 = calculate_ela_distribution(X_valid, y1_valid)
            
            # For objective 2  
            ela_meta_2 = calculate_ela_meta(X_valid, y2_valid)
            ela_dist_2 = calculate_ela_distribution(X_valid, y2_valid)
            
            # Average the landscape characteristics
            ela_results.append({
                'Property_1': prop1,
                'Property_2': prop2,
                'Correlation': row['Correlation'],
                'N_samples': len(y1_valid),
                'ELA_meta_lin_simple_avg': (ela_meta_1.get('ela_meta.lin_simple.adj_r2', 0) + 
                                             ela_meta_2.get('ela_meta.lin_simple.adj_r2', 0)) / 2,
                'ELA_meta_quad_simple_avg': (ela_meta_1.get('ela_meta.quad_simple.adj_r2', 0) + 
                                              ela_meta_2.get('ela_meta.quad_simple.adj_r2', 0)) / 2,
                'ELA_dist_skewness_avg': (ela_dist_1.get('ela_distr.skewness', 0) + 
                                          ela_dist_2.get('ela_distr.skewness', 0)) / 2,
                'ELA_dist_kurtosis_avg': (ela_dist_1.get('ela_distr.kurtosis', 0) + 
                                          ela_dist_2.get('ela_distr.kurtosis', 0)) / 2,
                'Ruggedness_proxy': 1 - (ela_meta_1.get('ela_meta.lin_simple.adj_r2', 0) + 
                                         ela_meta_2.get('ela_meta.lin_simple.adj_r2', 0)) / 2
            })
            
            print(f"✓ Analyzed: {prop1[:30]} vs {prop2[:30]}")
            
        except Exception as e:
            print(f"✗ Error analyzing {prop1} vs {prop2}: {str(e)}")
            continue
    
    if ela_results:
        ela_df = pd.DataFrame(ela_results)
        print("\nELA Features Summary:")
        print("=" * 80)
        print(ela_df.to_string(index=False))
        
        # Higher ruggedness = harder problem
        print("\nNote: Higher 'Ruggedness_proxy' indicates more complex landscape (harder optimization)")
    else:
        print("No ELA results computed.")
        ela_df = pd.DataFrame()

In [ ]:
# Cell 4: Step 3 - Pareto Front Analysis
print("\n" + "=" * 80)
print("STEP 3: PARETO FRONT ANALYSIS")
print("=" * 80)

if not pymoo_available:
    print("\nSkipping Pareto analysis - pymoo not installed.")
    print("Install with: pip install pymoo --break-system-packages")
    pareto_results = []
else:
    # For Pareto front analysis, we'll use surrogate models on the existing data
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.preprocessing import StandardScaler
    
    pareto_results = []
    
    print("\nAnalyzing Pareto front characteristics for top 10 pairs...\n")
    
    for idx, row in top_10_pairs.head(5).iterrows():  # Analyze top 5 to save time
        prop1, prop2 = row['Property_1'], row['Property_2']
        
        # Get data
        X = df[input_cols].values
        y1 = df[prop1].values
        y2 = df[prop2].values
        
        # Remove NaN entries
        valid_idx = ~(np.isnan(y1) | np.isnan(y2)) & ~np.any(np.isnan(X), axis=1)
        X_valid = X[valid_idx]
        y1_valid = y1[valid_idx]
        y2_valid = y2[valid_idx]
        
        if len(y1_valid) < 50:
            continue
            
        try:
            # Find dominated/non-dominated solutions in existing data
            Y = np.column_stack([y1_valid, y2_valid])
            
            # Simple Pareto dominance check
            is_pareto = np.ones(len(Y), dtype=bool)
            for i in range(len(Y)):
                for j in range(len(Y)):
                    if i != j:
                        # Check if j dominates i (assuming minimization)
                        if np.all(Y[j] <= Y[i]) and np.any(Y[j] < Y[i]):
                            is_pareto[i] = False
                            break
            
            pareto_points = Y[is_pareto]
            n_pareto = len(pareto_points)
            pareto_fraction = n_pareto / len(Y)
            
            # Compute hypervolume (requires normalized objectives)
            Y_norm = (Y - Y.min(axis=0)) / (Y.max(axis=0) - Y.min(axis=0) + 1e-10)
            pareto_norm = Y_norm[is_pareto]
            
            # Reference point for hypervolume (slightly worse than nadir)
            ref_point = np.max(Y_norm, axis=0) + 0.1
            
            try:
                hv = HV(ref_point=ref_point)
                hypervolume = hv(pareto_norm)
            except:
                hypervolume = np.nan
            
            # Estimate convexity (rough approximation)
            if n_pareto > 2:
                # Sort pareto points by first objective
                sorted_idx = np.argsort(pareto_points[:, 0])
                sorted_pareto = pareto_points[sorted_idx]
                
                # Check if points are below the line connecting extremes (convex)
                if len(sorted_pareto) > 2:
                    p1, p2 = sorted_pareto[0], sorted_pareto[-1]
                    # Count points below the connecting line
                    below_line = 0
                    for p in sorted_pareto[1:-1]:
                        # Linear interpolation
                        t = (p[0] - p1[0]) / (p2[0] - p1[0] + 1e-10)
                        expected_y = p1[1] + t * (p2[1] - p1[1])
                        if p[1] < expected_y:  # Below line (convex for minimization)
                            below_line += 1
                    convexity_score = below_line / (len(sorted_pareto) - 2)
                else:
                    convexity_score = np.nan
            else:
                convexity_score = np.nan
            
            pareto_results.append({
                'Property_1': prop1,
                'Property_2': prop2,
                'Correlation': row['Correlation'],
                'N_Pareto_Points': n_pareto,
                'Pareto_Fraction': pareto_fraction,
                'Hypervolume': hypervolume,
                'Convexity_Score': convexity_score,
                'Non_Convexity': 1 - convexity_score if not np.isnan(convexity_score) else np.nan
            })
            
            print(f"✓ Analyzed: {prop1[:30]} vs {prop2[:30]}")
            print(f"   Pareto points: {n_pareto}/{len(Y)} ({pareto_fraction:.2%})")
            
        except Exception as e:
            print(f"✗ Error analyzing {prop1} vs {prop2}: {str(e)}")
            continue
    
    if pareto_results:
        pareto_df = pd.DataFrame(pareto_results)
        print("\nPareto Front Characteristics:")
        print("=" * 80)
        print(pareto_df.to_string(index=False))
        
        print("\nNote: Lower Pareto_Fraction often indicates harder problems")
        print("      Higher Non_Convexity indicates more complex trade-offs")
    else:
        print("No Pareto results computed.")
        pareto_df = pd.DataFrame()

In [ ]:
def normalize_feature(x):
    """Normalize feature to 0-1 scale"""
    if x.isna().all():
        return x
    x_min = x.min()
    x_max = x.max()
    if x_max - x_min < 1e-10:
        return x * 0  # All same value -> all zeros
    return (x - x_min) / (x_max - x_min)


def combine_difficulty_scores(top_10_pairs, ela_df, pareto_df):
    """Combine all difficulty metrics into final score"""
    
    print("=" * 80)
    print("STEP 5: COMBINED DIFFICULTY SCORING")
    print("=" * 80)
    
    # Start with correlation results
    final_df = top_10_pairs.copy()
    
    # Add ELA features if available
    if not ela_df.empty:
        merge_cols = ['Property_1', 'Property_2', 'Ruggedness_avg', 
                     'Landscape_Difficulty', 'Linear_R2_avg', 'Local_Optima_estimate']
        available_cols = ['Property_1', 'Property_2'] + [c for c in merge_cols[2:] if c in ela_df.columns]
        
        final_df = final_df.merge(
            ela_df[available_cols],
            on=['Property_1', 'Property_2'],
            how='left'
        )
        ela_available = True
        print("\n✓ ELA features merged")
    else:
        final_df['Ruggedness_avg'] = np.nan
        final_df['Landscape_Difficulty'] = np.nan
        final_df['Linear_R2_avg'] = np.nan
        final_df['Local_Optima_estimate'] = np.nan
        ela_available = False
        print("\n⚠ No ELA features available")
    
    # Add Pareto features if available
    if not pareto_df.empty:
        merge_cols = ['Property_1', 'Property_2', 'N_Pareto_Points',
                     'Pareto_Fraction', 'Non_Convexity', 'Pareto_Complexity']
        available_cols = ['Property_1', 'Property_2'] + [c for c in merge_cols[2:] if c in pareto_df.columns]
        
        final_df = final_df.merge(
            pareto_df[available_cols],
            on=['Property_1', 'Property_2'],
            how='left'
        )
        pareto_available = True
        print("✓ Pareto features merged")
    else:
        final_df['N_Pareto_Points'] = np.nan
        final_df['Pareto_Fraction'] = np.nan
        final_df['Non_Convexity'] = np.nan
        final_df['Pareto_Complexity'] = np.nan
        pareto_available = False
        print("⚠ No Pareto features available")
    
    # Normalize features for scoring (0-1 scale)
    final_df['Conflict_Norm'] = normalize_feature(final_df['Conflict_Score'])
    final_df['Ruggedness_Norm'] = normalize_feature(final_df['Ruggedness_avg'])
    final_df['Landscape_Diff_Norm'] = normalize_feature(final_df['Landscape_Difficulty'])
    final_df['Pareto_Size_Norm'] = normalize_feature(final_df['N_Pareto_Points'])
    final_df['Non_Convexity_Norm'] = normalize_feature(final_df['Non_Convexity'])
    final_df['Pareto_Comp_Norm'] = normalize_feature(final_df['Pareto_Complexity'])
    
    # Compute combined difficulty score (higher = harder)
    # Dynamic weighting based on available features
    if ela_available and pareto_available:
        # All features available
        weights = {
            'conflict': 0.25,
            'landscape': 0.25,
            'ruggedness': 0.20,
            'pareto_comp': 0.20,
            'non_convex': 0.10
        }
        
        final_df['Difficulty_Score'] = (
            final_df['Conflict_Norm'].fillna(0) * weights['conflict'] +
            final_df['Landscape_Diff_Norm'].fillna(0) * weights['landscape'] +
            final_df['Ruggedness_Norm'].fillna(0) * weights['ruggedness'] +
            final_df['Pareto_Comp_Norm'].fillna(0) * weights['pareto_comp'] +
            final_df['Non_Convexity_Norm'].fillna(0) * weights['non_convex']
        )
        scoring_method = "Full (Correlation + ELA + Pareto)"
        
    elif ela_available:
        # Only correlation and ELA
        weights = {
            'conflict': 0.35,
            'landscape': 0.35,
            'ruggedness': 0.30
        }
        
        final_df['Difficulty_Score'] = (
            final_df['Conflict_Norm'].fillna(0) * weights['conflict'] +
            final_df['Landscape_Diff_Norm'].fillna(0) * weights['landscape'] +
            final_df['Ruggedness_Norm'].fillna(0) * weights['ruggedness']
        )
        scoring_method = "Correlation + ELA"
        
    elif pareto_available:
        # Only correlation and Pareto
        weights = {
            'conflict': 0.40,
            'pareto_comp': 0.35,
            'non_convex': 0.25
        }
        
        final_df['Difficulty_Score'] = (
            final_df['Conflict_Norm'].fillna(0) * weights['conflict'] +
            final_df['Pareto_Comp_Norm'].fillna(0) * weights['pareto_comp'] +
            final_df['Non_Convexity_Norm'].fillna(0) * weights['non_convex']
        )
        scoring_method = "Correlation + Pareto"
        
    else:
        # Only correlation
        final_df['Difficulty_Score'] = final_df['Conflict_Norm']
        scoring_method = "Correlation only"
    
    print(f"\nScoring method: {scoring_method}")
    
    # Sort by difficulty
    final_df_sorted = final_df.sort_values('Difficulty_Score', ascending=False)
    
    return final_df_sorted, scoring_method


def visualize_results(final_df_sorted, scoring_method):
    """Create visualizations of difficulty rankings"""
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # 1. Main difficulty ranking
    ax1 = fig.add_subplot(gs[0:2, 0])
    y_pos = np.arange(len(final_df_sorted))
    labels = [f"{row['Property_1'][:22]}\nvs\n{row['Property_2'][:22]}" 
              for _, row in final_df_sorted.iterrows()]
    
    colors = plt.cm.RdYlGn_r(final_df_sorted['Difficulty_Score'] / 
                             final_df_sorted['Difficulty_Score'].max())
    bars = ax1.barh(y_pos, final_df_sorted['Difficulty_Score'], color=colors, alpha=0.8)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(labels, fontsize=7)
    ax1.set_xlabel('Difficulty Score (Higher = Harder)', fontsize=10, fontweight='bold')
    ax1.set_title(f'Optimization Problem Difficulty Ranking\n({scoring_method})', 
                 fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, score) in enumerate(zip(bars, final_df_sorted['Difficulty_Score'])):
        ax1.text(score + 0.01, i, f'{score:.3f}', 
                va='center', fontsize=7)
    
    # 2. Correlation vs Difficulty scatter
    ax2 = fig.add_subplot(gs[0, 1])
    scatter = ax2.scatter(final_df_sorted['Correlation'], 
                         final_df_sorted['Difficulty_Score'],
                         c=final_df_sorted['Difficulty_Score'],
                         cmap='RdYlGn_r', s=100, alpha=0.7, edgecolors='black')
    ax2.set_xlabel('Correlation', fontweight='bold')
    ax2.set_ylabel('Difficulty Score', fontweight='bold')
    ax2.set_title('Correlation vs Difficulty', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax2, label='Difficulty')
    
    # 3. Feature contributions (if available)
    ax3 = fig.add_subplot(gs[1, 1])
    
    # Prepare data for stacked bar chart showing component contributions
    components = []
    component_labels = []
    
    if 'Conflict_Norm' in final_df_sorted.columns:
        components.append(final_df_sorted['Conflict_Norm'].fillna(0).values)
        component_labels.append('Conflict')
    
    if 'Ruggedness_Norm' in final_df_sorted.columns and not final_df_sorted['Ruggedness_Norm'].isna().all():
        components.append(final_df_sorted['Ruggedness_Norm'].fillna(0).values)
        component_labels.append('Ruggedness')
    
    if 'Pareto_Comp_Norm' in final_df_sorted.columns and not final_df_sorted['Pareto_Comp_Norm'].isna().all():
        components.append(final_df_sorted['Pareto_Comp_Norm'].fillna(0).values)
        component_labels.append('Pareto')
    
    if components:
        x = np.arange(len(final_df_sorted))
        bottom = np.zeros(len(final_df_sorted))
        
        colors_comp = plt.cm.Set3(np.linspace(0, 1, len(components)))
        
        for comp, label, color in zip(components, component_labels, colors_comp):
            ax3.barh(x, comp, left=bottom, label=label, color=color, alpha=0.8)
            bottom += comp
        
        ax3.set_yticks(x)
        ax3.set_yticklabels([f"Pair {i+1}" for i in range(len(final_df_sorted))], fontsize=7)
        ax3.set_xlabel('Normalized Component Contribution', fontweight='bold')
        ax3.set_title('Difficulty Components Breakdown', fontweight='bold')
        ax3.legend(fontsize=8)
        ax3.grid(True, alpha=0.3, axis='x')
    
    # 4. Summary statistics table
    ax4 = fig.add_subplot(gs[2, :])
    ax4.axis('off')
    
    # Create summary statistics
    summary_stats = []
    summary_stats.append(['Metric', 'Min', 'Mean', 'Max', 'Std'])
    summary_stats.append(['─' * 20] * 5)
    
    summary_stats.append([
        'Difficulty Score',
        f"{final_df_sorted['Difficulty_Score'].min():.3f}",
        f"{final_df_sorted['Difficulty_Score'].mean():.3f}",
        f"{final_df_sorted['Difficulty_Score'].max():.3f}",
        f"{final_df_sorted['Difficulty_Score'].std():.3f}"
    ])
    
    summary_stats.append([
        'Correlation',
        f"{final_df_sorted['Correlation'].min():.3f}",
        f"{final_df_sorted['Correlation'].mean():.3f}",
        f"{final_df_sorted['Correlation'].max():.3f}",
        f"{final_df_sorted['Correlation'].std():.3f}"
    ])
    
    if 'Ruggedness_avg' in final_df_sorted.columns and not final_df_sorted['Ruggedness_avg'].isna().all():
        summary_stats.append([
            'Ruggedness',
            f"{final_df_sorted['Ruggedness_avg'].min():.3f}",
            f"{final_df_sorted['Ruggedness_avg'].mean():.3f}",
            f"{final_df_sorted['Ruggedness_avg'].max():.3f}",
            f"{final_df_sorted['Ruggedness_avg'].std():.3f}"
        ])
    
    if 'Pareto_Fraction' in final_df_sorted.columns and not final_df_sorted['Pareto_Fraction'].isna().all():
        summary_stats.append([
            'Pareto Fraction',
            f"{final_df_sorted['Pareto_Fraction'].min():.3f}",
            f"{final_df_sorted['Pareto_Fraction'].mean():.3f}",
            f"{final_df_sorted['Pareto_Fraction'].max():.3f}",
            f"{final_df_sorted['Pareto_Fraction'].std():.3f}"
        ])
    
    # Draw table
    table = ax4.table(cellText=summary_stats, cellLoc='center', loc='center',
                     colWidths=[0.3, 0.15, 0.15, 0.15, 0.15])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Style header row
    for i in range(5):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    plt.tight_layout()
    plt.savefig('difficulty_ranking_comprehensive.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved comprehensive visualization to 'difficulty_ranking_comprehensive.png'")
    plt.show()


def print_summary(final_df_sorted):
    """Print final summary and recommendations"""
    
    print("\n" + "=" * 80)
    print("FINAL RESULTS & RECOMMENDATIONS")
    print("=" * 80)
    
    # Display full results table
    print("\nComplete Difficulty Rankings:")
    print("=" * 80)
    
    display_cols = ['Property_1', 'Property_2', 'Correlation', 'Difficulty_Score']
    
    # Add available metrics
    if 'Ruggedness_avg' in final_df_sorted.columns and not final_df_sorted['Ruggedness_avg'].isna().all():
        display_cols.append('Ruggedness_avg')
    if 'N_Pareto_Points' in final_df_sorted.columns and not final_df_sorted['N_Pareto_Points'].isna().all():
        display_cols.append('Pareto_Fraction')
    if 'Landscape_Difficulty' in final_df_sorted.columns and not final_df_sorted['Landscape_Difficulty'].isna().all():
        display_cols.append('Landscape_Difficulty')
    
    print(final_df_sorted[display_cols].to_string(index=False))
    
    # Hardest problem
    hardest = final_df_sorted.iloc[0]
    print("\n" + "=" * 80)
    print("🏆 HARDEST OPTIMIZATION PROBLEM IDENTIFIED:")
    print("=" * 80)
    print(f"  Objective 1: {hardest['Property_1']}")
    print(f"  Objective 2: {hardest['Property_2']}")
    print(f"  Correlation: {hardest['Correlation']:.3f}")
    print(f"  Difficulty Score: {hardest['Difficulty_Score']:.3f}")
    
    if 'Ruggedness_avg' in hardest and not pd.isna(hardest['Ruggedness_avg']):
        print(f"  Landscape Ruggedness: {hardest['Ruggedness_avg']:.3f}")
    if 'Pareto_Fraction' in hardest and not pd.isna(hardest['Pareto_Fraction']):
        print(f"  Pareto Fraction: {hardest['Pareto_Fraction']:.2%}")
    
    # Top 5 for research
    print("\n" + "=" * 80)
    print("📊 TOP 5 HARDEST PROBLEMS FOR OPTIMIZATION RESEARCH:")
    print("=" * 80)
    
    for i, (_, row) in enumerate(final_df_sorted.head(5).iterrows(), 1):
        print(f"\n{i}. {row['Property_1']}")
        print(f"   vs {row['Property_2']}")
        print(f"   └─ Correlation: {row['Correlation']:.3f}, Difficulty: {row['Difficulty_Score']:.3f}")
    
    # Research recommendations
    print("\n" + "=" * 80)
    print("💡 RESEARCH RECOMMENDATIONS:")
    print("=" * 80)
    print("""
    Based on this analysis, for your optimization speedup research:
    
    1. START WITH: The top-ranked pair (highest difficulty score)
       → This represents the most challenging optimization problem
       → Any speedup here will be most impressive
    
    2. VALIDATE WITH: The 2nd and 3rd ranked pairs
       → Ensures your method works across different hard problems
       → Provides robustness to your findings
    
    3. COMPARE AGAINST: A mid-difficulty pair (rank ~5-7)
       → Shows your method's performance across difficulty spectrum
       → Demonstrates when speedup is most beneficial
    
    4. KEY CHARACTERISTICS OF HARD PROBLEMS:
       → Highly conflicting objectives (correlation < -0.7)
       → Rugged landscapes (high nonlinearity)
       → Complex Pareto fronts (non-convex, sparse)
    """)


if __name__ == "__main__":
    # Load all results
    top_10_pairs = pd.read_csv('top_10_pairs.csv')
    
    # Load ELA results if available
    if os.path.exists('ela_results.csv'):
        ela_df = pd.read_csv('ela_results.csv')
    else:
        ela_df = pd.DataFrame()
    
    # Load Pareto results if available
    if os.path.exists('pareto_results.csv'):
        pareto_df = pd.read_csv('pareto_results.csv')
    else:
        pareto_df = pd.DataFrame()
    
    # Combine scores
    final_df_sorted, scoring_method = combine_difficulty_scores(top_10_pairs, ela_df, pareto_df)
    
    # Visualize
    visualize_results(final_df_sorted, scoring_method)
    
    # Print summary
    print_summary(final_df_sorted)
    
    # Save final results
    final_df_sorted.to_csv('optimization_difficulty_final.csv', index=False)
    print("\n" + "=" * 80)
    print("✓ Final results saved to 'optimization_difficulty_final.csv'")
    print("✓ Visualization saved to 'difficulty_ranking_comprehensive.png'")
    print("=" * 80)